In [0]:
checkpointLocation='/Volumes/test_catalog/landing/checkpoints'

df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .option("cloudFiles.schemaLocation", "/Volumes/test_catalog/landing/schemas")
         .load("/Volumes/test_catalog/landing/lookups/")
)

(
    df.writeStream
      .option("checkpointLocation",checkpointLocation)
      .trigger(availableNow=True)
      .toTable("test_catalog.learning.customer_test1")
)

In [0]:
%sql
select * from test_catalog.learning.customer_test1

In [0]:
%sql
describe history test_catalog.learning.customer_test1

In [0]:
%python
df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("/Volumes/test_catalog/landing/lookups/")
)

df.write.insertInto("test_catalog.learning.customer_test2")

In [0]:
%sql
OPTIMIZE test_catalog.learning.customer_test2
ZORDER BY (customer_id);

In [0]:
%sql
DESCRIBE HISTORY test_catalog.learning.customer_test2


In [0]:
%sql
OPTIMIZE test_catalog.learning.customer_test2
ZORDER BY (customer_id);

In [0]:
%sql
delete from test_catalog.learning.customer_test2 where customer_id in ('CUST00010','CUST00010')

In [0]:
%sql
SELECT DISTINCT _metadata.file_path
FROM test_catalog.learning.customer_test2;

In [0]:
%sql
alter table test_catalog.learning.customer_test2
cluster BY (customer_id);

In [0]:
%sql
select * from test_catalog.external_schema.test_external

In [0]:
%python
df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("/Volumes/test_catalog/landing/external")
)

df.write.insertInto(" test_catalog.external_schema.test_external")

In [0]:
%sql
DESCRIBE HISTORY test_catalog.external_schema.test_external

In [0]:
%sql
drop table test_catalog.external_schema.test_external

In [0]:
%sql
UNDROP TABLE test_catalog.external_schema.test_external;

In [0]:
%sql
select * from test_catalog.external_schema.test_external

In [0]:
%sql
drop table test_catalog.learning.customer_test1

In [0]:
%sql
undrop  table test_catalog.learning.customer_test1

MAX_BY  versus ROW_NUMBER windows
**![](path)![](**path**)**

In [0]:
%python
df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("/Volumes/test_catalog/landing/lookups/")
)

df.write.insertInto("test_catalog.learning.customer_test3")

In [0]:
%sql
with cte as (select customer_id, max_by(customer_name, last_updated) as customer_name 
from test_catalog.learning.customer_test3 GROUP BY customer_id) 

select count(1) from cte;

In [0]:
%sql
with cte as (
select customer_id, row_number() over (partition by customer_id  order by  last_updated desc) as row_num 
from test_catalog.learning.customer_test3)
select count(1) from cte where row_num=1;

In [0]:
%sql
alter table test_catalog.learning.customer_test3
cluster by (customer_id)

In [0]:
%sql

describe history test_catalog.learning.customer_test3


In [0]:
%sql
optimize  test_catalog.learning.customer_test3

In [0]:
%sql
SELECT DISTINCT _metadata.file_path
FROM test_catalog.learning.customer_test3;

PWC Interview Question — Senior Data Engineer


In [0]:
df1=spark.sql("select count(1) order_cnt ,order_status from test_catalog.bronze.bronze_orders group by order_status")


In [0]:
df=spark.sql("select *  from test_catalog.bronze.bronze_orders")
total=df.count()
print(total)

In [0]:
from pyspark.sql import functions as F

result_df = df1.withColumn(
    "percentage_of_total",
    F.concat(F.round((F.col("order_cnt") * 100) / total, 5), F.lit("%"))
)

display(result_df.orderBy(F.desc("order_cnt")))

In [0]:
%sql
SELECT
order_status
,COUNT(*) AS prder_count
,CONCAT(ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 5), '%') AS percentage_of_total
FROM test_catalog.bronze.bronze_orders
GROUP BY order_status
ORDER BY prder_count DESC;